# 00 — AIDP standard catalog setup

Create `lakehouse_demo` only if absent, confirm that it is a **standard** catalog,
and create the empty `bronze`, `silver`, and `gold` schemas if absent. Existing
objects are preserved. This notebook does not register tables or write source data.

Import this notebook into an authenticated AIDP workspace and attach a running
Spark compute cluster. Install the base project wheel on that cluster, or set
`REPO_ROOT` below to an accessible source checkout; no optional OCI dependency is needed.
Bronze v2 migration gate: run the separate migration prepare/apply/validation cell first. Do not run registration or Silver cells against the existing v1 registrations. See `docs/bronze-v2-migration.md`.


## Configuration

This is the only environment configuration section. Keep it identical in `01_register_bronze`.
The source bucket is shared; the standard catalog's own managed placement is selected by AIDP,
not by a `LOCATION` on the source bucket. Do not mount the shared source root as a volume.

In [ ]:
# CONFIGURATION: edit environment values only here, in both notebooks.
# These are non-secret values verified in docs/object-storage-execution.md.
CONFIG = {
    "catalog": "lakehouse_demo",
    "namespace": "frgpslcqn7sm",  # Object Storage namespace, NOT oraemeaaidata2
    "bucket": "CristinaVaras-bucket-AIDP",
    "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
    "reference_date": "2026-10-28",  # Fixed synthetic-data reference date
    "seed": 42,
    "batch_id": "initial",         # Do not include incremental in this milestone
}
# Optional: absolute driver-visible checkout path when using source imports.
# Leave None when the project wheel is installed on the AIDP compute cluster.
REPO_ROOT = None


In [ ]:
from pathlib import Path
import sys

if REPO_ROOT is not None:
    source_dir = Path(REPO_ROOT) / "src"
    if not (source_dir / "aidp_lakehouse" / "aidp_bronze.py").is_file():
        raise RuntimeError("REPO_ROOT must identify a driver-visible checkout with src/aidp_lakehouse")
    sys.path.insert(0, str(source_dir))

from aidp_lakehouse.aidp_bronze import BronzeConfig

cfg = BronzeConfig(**CONFIG)
if "spark" not in globals():
    raise RuntimeError("Attach authenticated AIDP compute and run with its supplied spark session")
print("Spark runtime:", spark.version)
print("Configured source root (not an ingestion wildcard):", cfg.root_uri)


## Inspect and create only absent metadata

The statements below are AIDP SQL, dispatched through the supplied `spark.sql` session.
A plain local Spark/Hive session cannot substitute for AIDP catalog DDL.
Oracle's sample uses the minimal standard-catalog CREATE form without connection OPTIONS.
An existing non-standard catalog with this name causes a stop before schema creation.

In [ ]:
from aidp_lakehouse.aidp_bronze import catalog_ddl, ensure_catalog

spark.sql(f"SHOW CATALOGS LIKE '{cfg.catalog}'").show(truncate=False)
for statement in catalog_ddl(cfg):
    print(statement + ";")
ensure_catalog(spark, cfg)


## Verify the resulting catalog and empty schema containers

DESCRIBE and SHOW are read-only. Existing child objects, if any, remain untouched.
Only successful AIDP execution of these cells establishes runtime verification.

In [ ]:
spark.sql(f"DESCRIBE CATALOG {cfg.catalog}").show(truncate=False)
spark.sql(f"SHOW SCHEMAS IN {cfg.catalog}").show(truncate=False)
for schema_name in ("bronze", "silver", "gold"):
    spark.sql(f"DESCRIBE SCHEMA {cfg.catalog}.{schema_name}").show(truncate=False)
print("Setup verification completed in this attached AIDP session.")


## Official references and execution status

DDL follows [Oracle AIDP SQL Grammar](https://docs.oracle.com/en/cloud/paas/ai-data-platform/aidug/sql-grammar.html),
[external-table support and limitations](https://docs.oracle.com/en/cloud/paas/ai-data-platform/aidug/tables.html),
and Oracle's [standard-catalog notebook sample](https://github.com/oracle-samples/oracle-aidp-samples/blob/main/data-engineering/transformation/liquid-clustering/financial_services_delta_liquid_clustering_demo.ipynb).
See `docs/aidp-bronze-setup.md` for imports, permissions and source contracts.

**Runtime verification: PENDING.** No cells in this checked-in notebook have been executed against AIDP.
No local OCI key or SQLcl Autonomous connection is used by this notebook.